# Práctica 1: Aprendizaje Automático
**Predicción de Subscripsción a un Producto Bancario**

* **Autores:** Jaime del Campo Montaña & David Alexander Torres Paillacho
* **NIA's:** 100495957 100522187

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, RobustScaler, OneHotEncoder, FunctionTransformer, MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree


RANDOM_STATE = 100522187
np.random.seed(RANDOM_STATE)


In [ ]:
dataframe = pd.read_pickle('bank_15.pkl')
#muestro las primeras filas del dataframe --> comprobación
display(dataframe.head())

#flas, columnas, tipos de datos y valores nulos
print("\nInformación del dataframe")
dataframe.info()

A continuación analizaremos los datos del dataset


In [ ]:
#número de instancias y variables
print(f"número de instancias (filas): {dataframe.shape[0]}")
print(f"número de variables (columnas): {dataframe.shape[1]}")
#comprobación de valores nulos
print("\nvalores null por columna")
nulos = dataframe.isnull().sum()
print(nulos[nulos > 0])

In [ ]:
#variables categóricas > 10
categorical_columans = dataframe.select_dtypes(include=['object']).columns
print("\nCardinalidad de variables categóricas")
for columna in categorical_columans:
    val_unicos = dataframe[columna].nunique()
    if val_unicos > 10:
        print(f"'{columna}'tiene alta cardinal              idad: {val_unicos} valores únicos")
    else:
        print(f"'{columna}':{val_unicos} valores únicos")

#columnas constantes
print("\ncolumnas constantes")
constant_columnas = [columna for columna in dataframe.columns if dataframe[columna].nunique() <= 1]
if len(constant_columnas) == 0:
    print("no hay columnas constantes.")
else:
    print(f"columnas constantes encontradas: {constant_columnas}")

In [ ]:
#comprobar desbalanceo de la variable objetivo
print("Distribución de la variable objetivo (deposit):")
print(dataframe['deposit'].value_counts(normalize=True))

#análisis de pdays
print("\nEstadísticas de pdays:")
print(dataframe['pdays'].describe())


In [ ]:
#separar X e y
X = dataframe.drop('deposit', axis=1)
y = dataframe['deposit']

#división Holdout (2/3 train, 1/3 test) según instrucciones
X_train, X_test, y_train, y_test = train_test_split(
X, y, test_size=0.33, random_state=RANDOM_STATE, stratify=y
)

print(f"Tamaño entrenamiento: {X_train.shape[0]}")
print(f"Tamaño test: {X_test.shape[0]}")

In [ ]:
cv_inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring_metric = 'roc_auc'

In [ ]:
#separamos columnas por tipo
#variables numéricas estándar 
num_cols = ['age', 'balance', 'day', 'duration', 'campaign', 'previous']

#variables categóricas con nulos
cat_cols_con_nulos = ['job', 'education']

#resto
cat_cols = ['marital', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome'] 

#pdays
pdays_col = ['pdays']


#transformacion de pdays
def transform_pdays(X):
    #transform_pdays recibe un array 2D, pero nosotros queremos trabajar con un array 1D para hacer las transformaciones más fácilmente.    
   
    pdays_valores = np.array(X).flatten() #aseguramos que es un array 1D
    
    no_previo = (pdays_valores == -1).astype(int) #1 si no hay contacto previo, 0 si sí lo hubo
    dias_validos = np.where(pdays_valores == -1, 0, pdays_valores) #reemplazamos -1 por 0 para que no afecte numéricamente
    return np.column_stack((no_previo, dias_validos))

def get_pdays_names(transformer, feature_names):
    return ['pdays_no_previo', 'pdays_dias_validos']

pdays_transformer = FunctionTransformer(
    transform_pdays, 
    validate=False,
    feature_names_out=get_pdays_names
)




#pipeline para las categóricas con nulos: Imputación + OneHotEncoding
cat_nulos_pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

#pipeline para el resto , solo el OneHotEncoding
cat_pipe = Pipeline(steps=[
    ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])


num_pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')) #por si hubiese algun valor nulo
])


#unimos todo en ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipe, num_cols),
        ('cat_nulos', cat_nulos_pipe, cat_cols_con_nulos),
        ('cat_normales', cat_pipe, cat_cols),
        ('pdays_custom', pdays_transformer, pdays_col)
    ],
    remainder='passthrough' 
)   

In [ ]:


#convertimos la variable objetivo a números 
y_train_num = y_train.map({'no': 0, 'yes': 1})
y_test_num = y_test.map({'no': 0, 'yes': 1})

#usamos StandardScaler como punto de partida para el escalado
knn_default_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier()) 
])

#medimos tiempo de entrenamiento directo
print("Entrenando KNN con hiperparámetros por defecto...")
inicio = time.time()
knn_default_pipe.fit(X_train, y_train_num)
final = time.time()
tiempo_default = final - inicio

#evaluamos usando la validación cruzada 
cv_scores_default = cross_val_score(knn_default_pipe, X_train, y_train_num, cv=cv_inner, scoring=scoring_metric, n_jobs=-1)

print(f"Tiempo de entrenamiento (Default): {tiempo_default:.4f} segundos")
print(f"Puntuación media ({scoring_metric}) en validación interna: {cv_scores_default.mean():.4f}")

In [ ]:
#creamos el pipeline
knn_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier())
])

#definimos la cuadricula de parametros
param_grid_knn = {
    'scaler': [StandardScaler(), MinMaxScaler(), RobustScaler()],
    'knn__n_neighbors': [3, 5, 7, 9, 11, 15, 17, 19, 21, 23, 25, 27, 29, 31, 33, 35]
}

#configuramos la busqueda
grid_knn = GridSearchCV(
    estimator=knn_pipeline,
    param_grid=param_grid_knn,
    scoring=scoring_metric, #roc_auc
    cv=cv_inner,            
    n_jobs=-1               
)

print("Entrenando y evaluando modelos KNN... espera un momento.")
#inicimaos cronómetro
inicio = time.time()
grid_knn.fit(X_train, y_train_num)
#finalizamos cronómetro
final = time.time()
tiempo = final - inicio
print("\nEntrenamiento completado")
print(f"El tiempo de ejucción del Gridsearch es {tiempo:.4f} segundos")
print(f"Mejor combinación encontrada: {grid_knn.best_params_}")
print(f"Mejor puntuación ({scoring_metric}) en validación interna: {grid_knn.best_score_:.4f}")

### 4.2 Árboles de Decisión 

A continuación, vamos a evaluar un modelo basado en Árboles de Decisión. Al igual que con KNN, primero entrenaremos un modelo con sus hiperparámetros por defecto para establecer una línea base y mediremos su tiempo de ejecución. 


In [ ]:
#creamos el pipeline del árbol 
tree_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('tree', DecisionTreeClassifier(random_state=RANDOM_STATE))])


print("Arbol por defecto")
inicio = time.time()
tree_pipeline.fit(X_train, y_train_num)
final = time.time()
tiempo = final - inicio

#evaluamos el inner del modelo por defecto
cv_scores_def_tree = cross_val_score(tree_pipeline, X_train, y_train_num, cv=cv_inner, scoring=scoring_metric, n_jobs=-1)

print(f"Tiempo de entrenamiento: {tiempo:.4f} segundos")
print(f"Puntuación media ({scoring_metric}) en validación interna: {cv_scores_def_tree.mean():.4f}\n")

#### Optimización de Hiperparámetros (HPO) para el Árbol

Para intentar mejorar el rendimiento base y evitar el sobreajuste, vamos a realizar una búsqueda en rejilla (GridSearchCV). Optimizaremos los siguientes hiperparámetros:
* max_depth:  controla la profundidad máxima del árbol.
* min_samples_split: número mínimo de muestras necesarias para dividir un nodo interno.
* criterion:  función para medir la calidad de una división.

In [ ]:
# Cuadrícula de hiperparámetros
param_grid_tree = {
    'tree__max_depth': [3, 5, 7, 10, None],
    'tree__min_samples_split': [2, 5, 10, 20],
    'tree__criterion': ['gini', 'entropy']
}

grid_tree = GridSearchCV(
    estimator=tree_pipeline,
    param_grid=param_grid_tree,
    scoring=scoring_metric, 
    cv=cv_inner,
    n_jobs=-1
)

start_time_hpo = time.time()
grid_tree.fit(X_train, y_train_num)
end_time_hpo = time.time()
tiempo_hpo_tree = end_time_hpo - start_time_hpo

print("¡Entrenamiento completado!")
print(f"Tiempo de ejecución del GridSearch: {tiempo_hpo_tree:.4f} segundos")
print(f"Mejor combinación: {grid_tree.best_params_}")
print(f"Mejor puntuación ({scoring_metric}) en validación interna: {grid_tree.best_score_:.4f}")

#### Interpretabilidad y Extracción de Conocimiento 

Una de las grandes ventajas de los árboles de decisión frente a métodos como KNN es que son modelos de "caja blanca". Para extraer conocimiento de negocio y entender qué variables son las más determinantes para que un cliente suscriba un depósito, vamos a entrenar un árbol poco profundo (*shallow tree* con max_depth=3) y lo visualizaremos.

In [ ]:
#entrenamos un árbol de profundidad 3 como ejemplo
shallow_tree_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('tree', DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE))
])
shallow_tree_pipe.fit(X_train, y_train_num)

#extraemos el modelo y los nombres de las columnas
modelo_arbol = shallow_tree_pipe.named_steps['tree']
nombres_features = shallow_tree_pipe.named_steps['preprocessor'].get_feature_names_out()

#dibujamos el árbol
plt.figure(figsize=(20, 10))
plot_tree(modelo_arbol, feature_names=nombres_features, 
          class_names=['No Deposit', 'Deposit'], filled=True, 
          rounded=True, fontsize=10)
plt.title("Shallow Tree (Profundidad=3) para Interpretabilidad")
plt.show()